# 0027 / 01 Source inventory

Scan the attached Top replay datasets from 2026-07-15 through 2026-08-01. This pass only commits source files and sizes so the attached days can be chosen after the volume is known.


In [ ]:
from __future__ import annotations
import csv, gzip, hashlib, json, re, zipfile
from collections import Counter
from datetime import date
from pathlib import Path
from typing import Any, Mapping

DATA_START = date(2026, 7, 15)
DATA_END = date(2026, 8, 1)
TARGET_DECK_SHA256 = "f50fa3a23cdf21be7cf7d3f558b8ff0b82e8d4e7ba8f61b7b4cacc1a0080c16a"
SCHEMA_VERSION = "0025_canonical_semantic_decision_v2"
DATE_RE = re.compile(r"20\d\d-\d\d-\d\d")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def write_csv(path: Path, rows: list[Mapping[str, Any]], fields: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

def path_date(path: Path) -> date | None:
    for value in DATE_RE.findall(str(path)):
        try:
            parsed = date.fromisoformat(value)
        except ValueError:
            continue
        if DATA_START <= parsed <= DATA_END:
            return parsed
    return None

def input_files(root: Path = Path("/kaggle/input")) -> list[Path]:
    return sorted(path for path in root.rglob("*") if path.is_file() and path_date(path) is not None)

def payloads(path: Path):
    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as bundle:
            for member in bundle.namelist():
                if not member.lower().endswith(".json"):
                    continue
                try:
                    value = json.loads(bundle.read(member))
                except Exception:
                    continue
                if isinstance(value, dict):
                    yield f"{path}!{member}", value
        return
    try:
        opener = gzip.open if path.name.lower().endswith(".gz") else open
        with opener(path, "rt", encoding="utf-8-sig") as handle:
            value = json.load(handle)
        if isinstance(value, dict):
            yield str(path), value
        elif isinstance(value, list):
            for index, item in enumerate(value):
                if isinstance(item, dict):
                    yield f"{path}#{index}", item
    except Exception:
        return

def first(value: Mapping[str, Any], *keys: str, default: Any = None) -> Any:
    for key in keys:
        if key in value and value[key] is not None:
            return value[key]
    return default

def as_list(value: Any) -> list[Any]:
    return value if isinstance(value, list) else []

def as_int(value: Any, default: int = 0) -> int:
    try:
        return int(value)
    except (TypeError, ValueError):
        return default

def episode_id(payload: Mapping[str, Any], fallback: str) -> str:
    info = payload.get("info")
    if isinstance(info, Mapping):
        value = first(info, "EpisodeId", "episode_id", "episodeId")
        if value is not None:
            return str(value)
    return str(first(payload, "episode_id", "episodeId", "id", default=fallback))

def winner(payload: Mapping[str, Any]) -> int | None:
    rewards = first(payload, "rewards", "reward", "scores", default=[])
    if not isinstance(rewards, list):
        return None
    indexes = [i for i, value in enumerate(rewards) if isinstance(value, (int, float)) and not isinstance(value, bool) and value > 0]
    return indexes[0] if len(indexes) == 1 else None

def deck_hash(cards: Any) -> str | None:
    ids = []
    for value in as_list(cards):
        if isinstance(value, Mapping):
            value = first(value, "id", "cardId")
        try:
            ids.append(int(value))
        except (TypeError, ValueError):
            return None
    if len(ids) != 60:
        return None
    return hashlib.sha256(",".join(map(str, sorted(ids))).encode("ascii")).hexdigest()

def frames(payload: Mapping[str, Any]):
    values = first(payload, "steps", "frames", "trajectory", "observations", default=[])
    for index, frame in enumerate(as_list(values)):
        if not isinstance(frame, Mapping):
            continue
        observation = first(frame, "observation", "obs", "state", default=frame)
        if not isinstance(observation, Mapping) or not isinstance(observation.get("select"), Mapping):
            continue
        current = observation.get("current") if isinstance(observation.get("current"), Mapping) else {}
        actor = as_int(first(frame, "playerIndex", "player", "actor", default=current.get("yourIndex", 0)))
        action = first(frame, "action", "selected", "ordered_action", "selection", default=[])
        yield index, actor, observation, action

OUT = Path("/kaggle/working/ptcg_0027_source_inventory")
rows = [{"date": str(path_date(path)), "path": str(path), "bytes": path.stat().st_size, "sha256": sha256_file(path)} for path in input_files()]
write_csv(OUT / "source_inventory.csv", rows, ["date", "path", "bytes", "sha256"])
write_json(OUT / "run_config.json", {"experiment": "0027_semantic_foundation_pretraining", "data_start": DATA_START.isoformat(), "data_end": DATA_END.isoformat(), "files": len(rows), "bytes": sum(row["bytes"] for row in rows), "selection": "date-bearing public replay files"})
print(json.dumps({"files": len(rows), "bytes": sum(row["bytes"] for row in rows), "output": str(OUT)}, indent=2))

